In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import TokenTextSplitter


from langchain_community.vectorstores import Chroma
from langchain_mistralai import MistralAIEmbeddings


from dotenv import load_dotenv
load_dotenv()

In [ ]:
data = PyMuPDFLoader("./testing.pdf")
docs = data.load()



In [ ]:
splitter = TokenTextSplitter(
    chunk_size=300,
    chunk_overlap=20
)

In [ ]:
chunks = splitter.split_documents(docs)

print(len(chunks))


#Model

In [ ]:
embedding_model = MistralAIEmbeddings(
    model="mistral-embed"
)

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

key = os.getenv("MISTRAL_API_KEY")

print("Key loaded:", bool(key))
print("Key length:", len(key) if key else 0)

In [ ]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)


In [ ]:
retriever = vector_store.as_retriever(
    search_type= "mmr",
    search_kwargs={
        "k": 3
    }
)

In [ ]:
result = retriever.invoke("What is the Real-world problem?")

In [ ]:
print(len(result))

In [ ]:
context = "\n\n".join(
    doc.page_content
    for doc in result
)

print(context)

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7
)

response = llm.invoke("What is NLP?")

print(response.content)

In [ ]:
question = "What is the real-world problem?"

In [ ]:
result = retriever.invoke(question)


In [ ]:
result = retriever.invoke(question)

for i, doc in enumerate(result):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content)

In [ ]:
context = "\n\n".join(
    doc.page_content
    
    for doc in result
)

In [ ]:
print(context)

In [ ]:
prompt = f"""
Answer the user's question using only the provided context.

Remove duplicate information and combine overlapping information.
Do not add information that is not present in the context.

Context:
{context}

Question:
{question}

Answer:
"""

In [ ]:
response = llm.invoke(prompt)


In [ ]:
print(response.content)